# Fashion-MNIST Deneyleri — Baseline (PixelCNN++) + ARPG

**Amaç:** Aynı dataset üzerinde klasik AR ile ARPG'yi karşılaştırıp raporlama verisi üretmek.

| Bölüm | Ne yapıyor | Tahmini süre (A100) |
|-------|-----------|---------------------|
| A | PixelCNN++ train + eval | ~2–3 saat |
| B | ARPG checkout + train + K-sweep | ~3–4 saat |
| C | Grafikler + özet tablo | ~15 dk |
| D (opsiyonel) | FID | ~30 dk |

**Not:** ARPG kodu `main`'de yok; Cell B1'de `origin/cifar` branch'ten checkout edilir (merge gerekmez).

In [ ]:
# Opsiyonel: Drive bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf COMP547PROJECT
!git clone https://github.com/oaydogdu/COMP547PROJECT.git
%cd COMP547PROJECT
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## A — Baseline: PixelCNN++ (Klassik AR)

In [ ]:
# CUDA hatasi aldiysan: Runtime -> Factory reset runtime (sadece Restart yetmez!)
%cd /content/COMP547PROJECT
!git pull origin main
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(torch.cuda.get_device_name(0))

!PYTHONPATH=src python scripts/train_pixelcnnpp.py \
  --dataset fashion_mnist \
  --epochs 20 \
  --batch-size 16 \
  --num-workers 0 \
  --save-dir results/pixelcnnpp_fashion_e20

In [ ]:
# Train bittikten sonra checkpoint var mi kontrol et
from pathlib import Path
%cd /content/COMP547PROJECT
ckpt_dir = Path("results/pixelcnnpp_fashion_e20/checkpoints")
print("CWD:", Path.cwd())
print("Checkpoints:", list(ckpt_dir.glob("*.pt")) or "(henuz yok — train devam ediyor veya basarisiz)")

In [ ]:
import glob, json, os
from pathlib import Path

%cd /content/COMP547PROJECT

save_dir = Path("results/pixelcnnpp_fashion_e20")
ckpt_dir = save_dir / "checkpoints"
ckpts = sorted(ckpt_dir.glob("*.pt"))

if not ckpts:
    print("Checkpoint bulunamadi. Once train cell'ini calistirin ve")
    print("'saved_checkpoint=...' ciktisini bekleyin.")
    print("Mevcut results klasoru:")
    !ls -R results 2>/dev/null || echo "(results yok)"
    raise FileNotFoundError(f"No checkpoint in {ckpt_dir}")

ckpt = str(ckpts[-1])
print("Checkpoint:", ckpt)

os.makedirs(save_dir / "eval", exist_ok=True)
!PYTHONPATH=src python scripts/eval_pixelcnnpp.py \
  --checkpoint "{ckpt}" \
  --out-json results/pixelcnnpp_fashion_e20/eval/fashion_eval.json \
  --out-grid results/pixelcnnpp_fashion_e20/eval/fashion_grid.png \
  --sample-batch-size 25

In [ ]:
import glob, json
from pathlib import Path
import matplotlib.pyplot as plt

%cd /content/COMP547PROJECT

metrics_files = sorted(Path("results/pixelcnnpp_fashion_e20/metrics").glob("*.json"))
if not metrics_files:
    raise FileNotFoundError("Metrics yok — train cell'i tamamlanmamis olabilir.")

hist = json.loads(metrics_files[-1].read_text())
best = min(hist, key=lambda x: x["test_bpd"])
print(f"Best test BPD: {best['test_bpd']:.4f} @ epoch {best['epoch']}")

epochs = [h["epoch"] for h in hist]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [h["train_bpd"] for h in hist], label="train")
plt.plot(epochs, [h["test_bpd"] for h in hist], label="test")
plt.xlabel("epoch"); plt.ylabel("BPD"); plt.legend(); plt.grid(alpha=0.3)
plt.title("PixelCNN++ Fashion-MNIST")
plt.savefig("results/pixelcnnpp_fashion_e20/eval/bpd_curve.png", dpi=150)
plt.show()

## B — ARPG (cifar branch'ten dosya checkout)

In [ ]:
%cd /content/COMP547PROJECT
!git fetch origin cifar
!git checkout origin/cifar -- src/ARPG/ scripts/train_arpg.py scripts/eval_arpg.py
!ls src/ARPG/arpg_model.py src/ARPG/arpg_runner.py scripts/train_arpg.py scripts/eval_arpg.py

In [ ]:
%cd /content/COMP547PROJECT
!PYTHONPATH=src python scripts/train_arpg.py \
  --dataset fashion_mnist \
  --data-dir data \
  --save-dir results/arpg_fashion \
  --epochs 20 \
  --batch-size 16 \
  --d-model 192 \
  --n-heads 6 \
  --n-layers 6 \
  --seed 1

In [ ]:
import glob
ckpt = sorted(glob.glob('results/arpg_fashion/checkpoints/*.pt'))[-1]
print('ARPG checkpoint:', ckpt)

!PYTHONPATH=src python scripts/eval_arpg.py \
  --checkpoint "{ckpt}" \
  --out-dir results/arpg_fashion/eval \
  --ks 1,2,4,7,14,28,56,112,196,392,784 \
  --schedules random,raster,row \
  --n-samples 25 \
  --seed 42 \
  --top-p 0.9 \
  --temperature 1.0

## C — Raporlama: Speed tradeoff + özet

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

with open('results/arpg_fashion/eval/sweep.json') as f:
    sweep = json.load(f)['sweep']

ar_json = 'results/pixelcnnpp_fashion_e20/eval/fashion_eval.json'
ar_lat = ar_tp = None
if os.path.exists(ar_json):
    ar = json.load(open(ar_json))
    ar_lat, ar_tp = ar['latency_ms_per_image'], ar['throughput_img_per_s']

schedules = ['random', 'raster', 'row']
colors  = {'random': '#2196F3', 'raster': '#FF9800', 'row': '#4CAF50'}
markers = {'random': 'o', 'raster': 's', 'row': '^'}
labels  = {'random': 'Random (ARPG)', 'raster': 'Raster', 'row': 'Row-by-row'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Fashion-MNIST: ARPG Speed Tradeoff vs AR Baseline', fontsize=14, fontweight='bold')

for sched in schedules:
    rows = [r for r in sweep if r['schedule'] == sched]
    Ks   = np.array([r['K'] for r in rows])
    lats = np.array([r['latency_ms_per_image'] for r in rows])
    tps  = np.array([r['throughput_img_per_s'] for r in rows])
    axes[0].plot(Ks, lats, marker=markers[sched], color=colors[sched], label=labels[sched], linewidth=2)
    axes[1].plot(Ks, tps,  marker=markers[sched], color=colors[sched], label=labels[sched], linewidth=2)

if ar_lat:
    axes[0].axhline(ar_lat, color='red', linestyle='--', linewidth=2,
                    label=f'AR Sequential (PixelCNN++) {ar_lat:.0f} ms')
    axes[1].axhline(ar_tp, color='red', linestyle='--', linewidth=2,
                    label=f'AR Sequential {ar_tp:.1f} img/s')

for ax in axes:
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlabel('K (decode steps) — düşük K = hızlı, yüksek K = kaliteli')
axes[0].set_ylabel('Latency (ms / image)')
axes[1].set_ylabel('Throughput (img / s)')
plt.tight_layout()
plt.savefig('results/arpg_fashion/eval/tradeoff_speed.png', dpi=180)
plt.show()
print('Saved: results/arpg_fashion/eval/tradeoff_speed.png')

In [ ]:
# Kalite seridi: her schedule için K artışıyla grid'ler
for sched in ['random', 'raster', 'row']:
    rows = [r for r in sweep if r['schedule'] == sched]
    fig, axes = plt.subplots(1, len(rows), figsize=(2.8 * len(rows), 3.2))
    if len(rows) == 1:
        axes = [axes]
    for ax, r in zip(axes, rows):
        ax.imshow(Image.open(r['grid']), cmap='gray')
        ax.axis('off')
        ax.set_title(f"K={r['K']} ({r['latency_ms_per_image']:.0f}ms)", fontsize=8)
    plt.suptitle(f'Schedule: {sched}', fontsize=12)
    plt.tight_layout()
    out = f'results/arpg_fashion/eval/quality_strip_{sched}.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print('Saved:', out)

In [ ]:
# Sunum/rapor için tek özet tablo
import pandas as pd

rows = []
for r in sweep:
    rows.append({
        'model': 'ARPG',
        'schedule': r['schedule'],
        'K': r['K'],
        'latency_ms': round(r['latency_ms_per_image'], 2),
        'throughput': round(r['throughput_img_per_s'], 3),
    })
if ar_lat:
    rows.append({
        'model': 'PixelCNN++',
        'schedule': 'sequential',
        'K': 784,
        'latency_ms': round(ar_lat, 2),
        'throughput': round(ar_tp, 3),
    })

df = pd.DataFrame(rows)
display(df)
df.to_csv('results/fashion_mnist_summary.csv', index=False)
print('Saved: results/fashion_mnist_summary.csv')

## D — Opsiyonel: FID (kalite metriği)

FID için en az ~1000 generated sample önerilir. Hızlı pilot için K=28 random ile 100 sample yeterli olabilir (raporda sample size belirt).

In [ ]:
# !pip install -q clean-fid
# TODO: generated sample'ları kaydet, clean-fid ile train set'e karşı FID hesapla
# Örnek: cleanfid.compute_fid(gen_dir, dataset_name='FashionMNIST', mode='clean', dataset_res=28)

In [ ]:
# Sonuçları Drive'a kopyala (opsiyonel)
!mkdir -p /content/drive/MyDrive/comp547_outputs/fashion_mnist
!cp -r results/pixelcnnpp_fashion_e20 results/arpg_fashion results/fashion_mnist_summary.csv \
       /content/drive/MyDrive/comp547_outputs/fashion_mnist/
print('Drive kaydı tamam.')